In [140]:
import time

import requests
import os
from dotenv import load_dotenv
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

load_dotenv()

account_name = os.getenv("VTEX_ACCOUNT_NAME")
app_key = os.getenv("VTEX_APP_KEY")
app_token = os.getenv("VTEX_APP_TOKEN")
headers = {
    'Accept': 'application/json',
	'Content-Type': 'application/json',
	'X-VTEX-API-AppKey': app_key,
	'X-VTEX-API-AppToken': app_token
}
BASE_URL = f"https://{account_name}.vtexcommercestable.com.br/api/catalog_system/pvt"

In [141]:
@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    retry=retry_if_exception_type(requests.exceptions.RequestException)
)
def vtex_get(path, params=None):
    """Função genérica para requisições GET na VTEX."""
    url = f"{BASE_URL}/{path}"
    time.sleep(1)
    response = requests.get(url, headers=headers, params=params, timeout=5)
    response.raise_for_status()
    return response.json()

In [142]:
# Vtex funcions
def total_ids():
    try:
        res = vtex_get("products/GetProductAndSkuIds")
        return res['range']['total']
    except (requests.exceptions.RequestException, KeyError) as e:
        print(f"Não foi possível capturar todos os Ids: {e}")
        return None

def sku_id_list(start, end):
    try:
        res = vtex_get("products/GetProductAndSkuIds", params={'_from': start, '_to': end})
        return res['data']
    except requests.exceptions.RequestException as e:
        print(f"Não foi possível capturar a lista de Ids: {e}")
        return None

def fetch_refid_using_product_id(productid):
    try:
        res = vtex_get(f"sku/stockkeepingunitByProductId/{productid}")
        return res[0]['RefId']
    except (requests.exceptions.RequestException, IndexError, KeyError) as e:
        print(f"Não foi possível capturar o Refid do product {productid}: {e}")
        return None

def fetch_sku_ids(salto, total_sku_ids):
    inicio = 1
    dict_sku_ids = {}

    try:
        while inicio < total_sku_ids:
            limite = inicio + salto
            if limite < total_sku_ids:
                dict_sku_ids.update(sku_id_list(inicio, limite-1))
                inicio += salto
            else:
                dict_sku_ids.update(sku_id_list(inicio, total_sku_ids))
                inicio += salto
    except Exception as e:
        print(f"Não foi possível capturar todos os Sku/Ids: {e}")
        return {}

    return dict_sku_ids

In [143]:
def complete_refid_list():
    refid_list = []
    total = total_ids()
    print(f"🔢 Total de produtos: {total}")
    dict_sku_ids = fetch_sku_ids(250, total)

    for key, value in dict_sku_ids.items():
        try:
            ref_id = fetch_refid_using_product_id(key)
            print(f"Salvando Product Id: {key}, Sku (Id): {value[0]}, RefId: {ref_id}")
            refid_list.append(ref_id)
        except Exception as e:
            print(f"❌ Falha ao buscar RefID para Product ID {key}: {e}")

    return refid_list

complete_refid_list()